# 02 — Geocode container addresses

This notebook takes the cleaned LIMASAM addresses and geocodes each one using Nominatim (OpenStreetMap), producing point coordinates ready for spatial analysis.

**Input:** `data/interim/containers_cleaned.csv` (480 cleaned addresses)
**Output:** `data/processed/containers_geocoded.csv` (with `lat`, `lon`, `match_type`, `geocoding_quality`)

## Strategy

1. Use Nominatim, respecting its rate limit (1 request per second) and policy (custom user-agent).
2. For each address, capture the returned `match_type` (`house` / `road` / `place` etc.) to assess geocoding quality.
3. Tag each result as:
   - **`high`** — `house_number` request returned a precise `house` or `building` match
   - **`medium`** — returned a street or intersection match
   - **`low`** — only a generic place or POI was found
   - **`failed`** — no result
4. Validate that all points fall inside Málaga's bounding box; flag any outliers.

## Notes

- ~480 requests × 1 second = ~8–10 minutes runtime.
- Results are cached: if you re-run the notebook, addresses already geocoded are skipped.

In [1]:
import time
from pathlib import Path

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)  # create if missing

# Files
INPUT_PATH = DATA_INTERIM / "containers_cleaned.csv"
OUTPUT_PATH = DATA_PROCESSED / "containers_geocoded.csv"

print(f"Input file:  {INPUT_PATH}")
print(f"Output file: {OUTPUT_PATH}")
print(f"Input exists: {INPUT_PATH.exists()}")

Input file:  c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\containers_cleaned.csv
Output file: c:\Users\user\Documents\Projects\malaga-textile-access\data\processed\containers_geocoded.csv
Input exists: True


In [15]:
df = pd.read_csv(INPUT_PATH)

print(f"Loaded {len(df)} addresses")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

Loaded 478 addresses
Columns: ['address_raw', 'address_type', 'address_clean']

First 3 rows:


,address_raw,address_type,address_clean
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...",house_number,"Avenida de la Paloma, 36-38, Carretera de Cádi..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...",house_number,"Avenida de la Paloma, 43, Carretera de Cádiz, ..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...",house_number,"Calle Conde de Guadalhorce, 6-8, Cruz de Humil..."


In [3]:
# Set up Nominatim with a custom user-agent (their policy requires identifying the app)
geolocator = Nominatim(
    user_agent="malaga-textile-access-project (lidavaynberg@gmail.com)",
    timeout=10,
)

# Wrap in a rate limiter: at most 1 request per second
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,   # 1.1s to be safe
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False,
)

# Pick one address from each type to test
test_samples = (
    df.groupby("address_type")
      .first()
      .reset_index()
)
print("Testing geocoder on one address of each type:\n")

for _, row in test_samples.iterrows():
    addr = row["address_clean"]
    addr_type = row["address_type"]
    print(f"[{addr_type}]  {addr}")
    
    location = geocode(addr)
    if location:
        print(f"  → ({location.latitude:.5f}, {location.longitude:.5f})")
        print(f"  → matched: {location.address}")
        print(f"  → raw type: {location.raw.get('type')}, class: {location.raw.get('class')}")
    else:
        print(f"  → NOT FOUND")
    print()

Testing geocoder on one address of each type:

[house_number]  Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  → NOT FOUND

[intersection]  Andarax and Camino de San Rafael, Málaga, Spain
  → NOT FOUND

[street_only]  Calle Almogía, Málaga, Spain
  → (36.71439, -4.45309)
  → matched: Calle Almogía, Polígono Carretera de Cártama, Los Corazones, Cruz de Humilladero, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29010, España
  → raw type: secondary, class: highway



In [4]:
# Test variations for a house_number address
test_address_variants = [
    "Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain",
    "Avenida de la Paloma, 36, Carretera de Cádiz, 29003, Málaga, Spain",
    "Avenida de la Paloma, 36, 29003 Málaga, Spain",
    "Avenida de la Paloma, 36, Málaga, Spain",
    "Avenida de la Paloma 36, Málaga, Spain",
    "Av de la Paloma 36, Málaga",
]

print("Testing variations of the same address:\n")
for variant in test_address_variants:
    print(f"→ {variant}")
    loc = geocode(variant)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}")
    else:
        print(f"  ✗ NOT FOUND")
    print()

Testing variations of the same address:

→ Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, Carretera de Cádiz, 29003, Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, 29003 Málaga, Spain
  ✓ (36.32259, -5.25230)  type=residential

→ Avenida de la Paloma, 36, Málaga, Spain
  ✓ (36.32259, -5.25230)  type=residential

→ Avenida de la Paloma 36, Málaga, Spain
  ✓ (36.32259, -5.25230)  type=residential

→ Av de la Paloma 36, Málaga
  ✓ (36.49128, -4.70622)  type=residential



In [5]:
# Málaga bounding box (approximate, generous)
# (west, south, east, north) — covering the city and immediate surroundings
malaga_bbox = [(-4.55, 36.65), (-4.30, 36.78)]
#                ↑ SW corner       ↑ NE corner

def geocode_in_malaga(address):
    """Geocode an address, restricting results to Málaga's bounding box and Spain."""
    return geocode(
        address,
        country_codes="es",          # only Spain
        viewbox=malaga_bbox,
        bounded=True,                 # strict: ONLY return matches inside the box
    )

# Re-test the same variations, now restricted to Málaga
test_address_variants = [
    "Avenida de la Paloma, 36, 29003 Málaga, Spain",
    "Avenida de la Paloma, 36, Málaga, Spain",
    "Avenida de la Paloma 36, Málaga, Spain",
    "Av de la Paloma 36, Málaga",
]

print("Re-testing with Málaga bounding box restriction:\n")
for variant in test_address_variants:
    print(f"→ {variant}")
    loc = geocode_in_malaga(variant)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}")
        print(f"    matched: {loc.address[:100]}...")
    else:
        print(f"  ✗ NOT FOUND")
    print()

Re-testing with Málaga bounding box restriction:

→ Avenida de la Paloma, 36, 29003 Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma 36, Málaga, Spain
  ✗ NOT FOUND

→ Av de la Paloma 36, Málaga
  ✗ NOT FOUND



In [6]:
# Search just the street, without house number
test_streets = [
    "Avenida de la Paloma, Málaga",
    "Calle Goya, Málaga",
    "Calle Conde de Guadalhorce, Málaga",
]

print("Testing street-only searches with bounding box:\n")
for street in test_streets:
    print(f"→ {street}")
    loc = geocode_in_malaga(street)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})")
        print(f"    matched: {loc.address[:120]}")
    else:
        print(f"  ✗ NOT FOUND inside Málaga bbox")
    print()

Testing street-only searches with bounding box:

→ Avenida de la Paloma, Málaga
  ✗ NOT FOUND inside Málaga bbox

→ Calle Goya, Málaga
  ✗ NOT FOUND inside Málaga bbox

→ Calle Conde de Guadalhorce, Málaga
  ✗ NOT FOUND inside Málaga bbox



In [7]:
# Test 1: search WITHOUT any restriction — does Nominatim find Calle Goya at all?
print("=== Test 1: no restrictions ===")
loc = geocode("Calle Goya, Málaga, Spain")
if loc:
    print(f"  Found at: ({loc.latitude:.5f}, {loc.longitude:.5f})")
    print(f"  Full address: {loc.address}")
else:
    print("  NOT FOUND")

# Test 2: same but only country restriction (no bbox)
print("\n=== Test 2: only country=es ===")
loc = geocode("Calle Goya, Málaga, Spain", country_codes="es")
if loc:
    print(f"  Found at: ({loc.latitude:.5f}, {loc.longitude:.5f})")
    print(f"  Full address: {loc.address}")
else:
    print("  NOT FOUND")

# Show what bbox I'm passing
print("\n=== Current bbox parameter ===")
print(f"malaga_bbox = {malaga_bbox}")
print(f"This is interpreted as: SW corner = {malaga_bbox[0]}, NE corner = {malaga_bbox[1]}")
print(f"  i.e. longitude {malaga_bbox[0][0]} to {malaga_bbox[1][0]}")
print(f"  and  latitude  {malaga_bbox[0][1]} to {malaga_bbox[1][1]}")

=== Test 1: no restrictions ===
  Found at: (36.70537, -4.43710)
  Full address: Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

=== Test 2: only country=es ===
  Found at: (36.70579, -4.43760)
  Full address: Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

=== Current bbox parameter ===
malaga_bbox = [(-4.55, 36.65), (-4.3, 36.78)]
This is interpreted as: SW corner = (-4.55, 36.65), NE corner = (-4.3, 36.78)
  i.e. longitude -4.55 to -4.3
  and  latitude  36.65 to 36.78


In [8]:
# CORRECT format: viewbox expects (latitude, longitude) tuples
# Málaga bounding box (generous, covers city + immediate surroundings)
malaga_bbox = [(36.65, -4.55), (36.78, -4.30)]
#                ↑ SW (lat, lon)   ↑ NE (lat, lon)

def geocode_in_malaga(address):
    """Geocode an address, restricting results to Málaga's bounding box and Spain."""
    return geocode(
        address,
        country_codes="es",
        viewbox=malaga_bbox,
        bounded=True,
    )

# Re-test
test_address_variants = [
    "Avenida de la Paloma, 36, 29003 Málaga, Spain",
    "Avenida de la Paloma, 36, Málaga, Spain",
    "Calle Goya, 7, Málaga, Spain",
    "Calle Goya, Málaga, Spain",
    "Calle Conde de Guadalhorce, 6, Málaga, Spain",
    "Andarax and Camino de San Rafael, Málaga, Spain",
    "Calle Almogía, Málaga, Spain",
]

print("Re-testing with FIXED bbox order:\n")
for variant in test_address_variants:
    print(f"→ {variant}")
    loc = geocode_in_malaga(variant)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  type={loc.raw.get('type')}, class={loc.raw.get('class')}")
        print(f"    {loc.address[:120]}")
    else:
        print(f"  ✗ NOT FOUND")
    print()

Re-testing with FIXED bbox order:

→ Avenida de la Paloma, 36, 29003 Málaga, Spain
  ✗ NOT FOUND

→ Avenida de la Paloma, 36, Málaga, Spain
  ✗ NOT FOUND

→ Calle Goya, 7, Málaga, Spain
  ✓ (36.70553, -4.43741)  type=house, class=place
    7, Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

→ Calle Goya, Málaga, Spain
  ✓ (36.70537, -4.43710)  type=tertiary, class=highway
    Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

→ Calle Conde de Guadalhorce, 6, Málaga, Spain
  ✓ (36.71332, -4.44156)  type=house, class=place
    6A, Calle Conde de Guadalhorce, Cruz del Humilladero, Cruz de Humilladero, Málaga, Málaga-Costa del Sol, Málaga, Andaluc

→ Andarax and Camino de San Rafael, Málaga, Spain
  ✗ NOT FOUND

→ Calle Almogía, Málaga, Spain
  ✓ (36.71424, -4.45314)  type=secondary, class=highway
    Calle Almogía, La Barriguilla, Cruz de Humilladero, Málaga, Málaga-Costa del Sol, M

In [9]:
import re

def simplify_address(address: str) -> str:
    """Remove 'de la', 'del', 'de' between street type and street name."""
    # Match patterns like 'Avenida de la X', 'Calle del Y', 'Calle de Z'
    # Be careful: only remove the article right after the street type
    return re.sub(
        r"\b(Avenida|Calle|Plaza|Camino|Carretera|Pasaje|Paseo|Glorieta)\s+(de\s+la|de\s+las|del|de\s+los|de)\s+",
        r"\1 ",
        address,
        flags=re.IGNORECASE,
    )


def strip_house_number(address: str) -> str:
    """Remove the first numeric block (house number) from an address."""
    # Pattern: ', 36-38, ...' or ', 36, ...' or ' 36, ...'
    return re.sub(r",?\s*\d+[\d\-A-Za-z]*\s*,", ",", address, count=1).strip(", ")


def geocode_with_fallbacks(address_clean: str, address_type: str):
    """Try several formats; return (location, quality_tag)."""
    
    # Attempt 1: original
    loc = geocode_in_malaga(address_clean)
    if loc:
        match_class = loc.raw.get("class")
        if match_class == "place" and loc.raw.get("type") == "house":
            return loc, "high", "original"
        else:
            return loc, "medium", "original"
    
    # Attempt 2: simplified (remove "de la", "del" etc.)
    simplified = simplify_address(address_clean)
    if simplified != address_clean:
        loc = geocode_in_malaga(simplified)
        if loc:
            match_class = loc.raw.get("class")
            if match_class == "place" and loc.raw.get("type") == "house":
                return loc, "high", "simplified"
            else:
                return loc, "medium", "simplified"
    
    # Attempt 3: strip house number, try as street
    no_number = strip_house_number(address_clean)
    if no_number != address_clean:
        loc = geocode_in_malaga(no_number)
        if loc:
            return loc, "low", "street_only_fallback"
    
    # Attempt 4: strip house number AND simplify
    no_num_simplified = simplify_address(strip_house_number(address_clean))
    if no_num_simplified not in (address_clean, simplified, no_number):
        loc = geocode_in_malaga(no_num_simplified)
        if loc:
            return loc, "low", "street_only_simplified_fallback"
    
    return None, "failed", "none"


# Test on the problematic samples
test_cases = [
    ("Avenida de la Paloma, 36, 29003, Málaga, Spain", "house_number"),
    ("Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain", "house_number"),
    ("Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Andarax and Camino de San Rafael, Málaga, Spain", "intersection"),
    ("Calle Almogía, Málaga, Spain", "street_only"),
]

print("Testing fallback geocoding strategy:\n")
for addr, addr_type in test_cases:
    print(f"[{addr_type}]  {addr}")
    loc, quality, method = geocode_with_fallbacks(addr, addr_type)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  quality={quality}, method={method}")
        print(f"    {loc.address[:120]}")
    else:
        print(f"  ✗ FAILED")
    print()

Testing fallback geocoding strategy:

[house_number]  Avenida de la Paloma, 36, 29003, Málaga, Spain
  ✓ (36.70166, -4.44376)  quality=high, method=simplified
    36, Avenida Paloma, San Carlos Condote, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, Espa

[house_number]  Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70066, -4.44184)  quality=medium, method=simplified
    Avenida Paloma, Girón, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, España

[house_number]  Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain
  ✓ (36.70553, -4.43741)  quality=high, method=original
    7, Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

[house_number]  Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70023, -4.43988)  quality=medium, method=original
    Calle Federico García Lorca, 25 Años de Paz, Carretera de Cádiz,

In [10]:
def split_intersection(address: str):
    """If the address contains ' and ', return the first and second street names."""
    # Look for ' and ' as our standardized intersection separator
    parts = re.split(r"\s+and\s+", address, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) == 2:
        # First part is street A; second part is "street B, Málaga, Spain" — strip the tail
        street_a = parts[0].strip()
        # Reconstruct street B with the Málaga suffix
        street_b_raw = parts[1].strip()
        # Take only the street name from street B (before the first comma)
        street_b_name = street_b_raw.split(",")[0].strip()
        return street_a + ", Málaga, Spain", street_b_name + ", Málaga, Spain"
    return None, None


def geocode_with_fallbacks(address_clean: str, address_type: str):
    """Try several formats; return (location, quality_tag, method_used)."""
    
    # Attempt 1: original
    loc = geocode_in_malaga(address_clean)
    if loc:
        match_class = loc.raw.get("class")
        match_type = loc.raw.get("type")
        if match_class == "place" and match_type == "house":
            return loc, "high", "original"
        elif address_type == "street_only":
            # Street-only addresses cannot do better than this
            return loc, "high", "original"  # best possible for this type
        else:
            return loc, "medium", "original"
    
    # Attempt 2: simplified (remove "de la", "del" etc.)
    simplified = simplify_address(address_clean)
    if simplified != address_clean:
        loc = geocode_in_malaga(simplified)
        if loc:
            match_class = loc.raw.get("class")
            match_type = loc.raw.get("type")
            if match_class == "place" and match_type == "house":
                return loc, "high", "simplified"
            elif address_type == "street_only":
                return loc, "high", "simplified"
            else:
                return loc, "medium", "simplified"
    
    # Attempt 3 (for intersections): try first street alone
    if address_type == "intersection":
        street_a, street_b = split_intersection(address_clean)
        if street_a:
            loc = geocode_in_malaga(street_a)
            if loc:
                return loc, "low", "first_street_of_intersection"
            # Try simplified first street
            loc = geocode_in_malaga(simplify_address(street_a))
            if loc:
                return loc, "low", "first_street_simplified"
            # Try second street
            if street_b:
                loc = geocode_in_malaga(street_b)
                if loc:
                    return loc, "low", "second_street_of_intersection"
                loc = geocode_in_malaga(simplify_address(street_b))
                if loc:
                    return loc, "low", "second_street_simplified"
    
    # Attempt 4 (for house_number): strip house number, try as street
    if address_type == "house_number":
        no_number = strip_house_number(address_clean)
        if no_number != address_clean:
            loc = geocode_in_malaga(no_number)
            if loc:
                return loc, "low", "street_only_fallback"
            no_num_simplified = simplify_address(no_number)
            if no_num_simplified != no_number:
                loc = geocode_in_malaga(no_num_simplified)
                if loc:
                    return loc, "low", "street_only_simplified_fallback"
    
    return None, "failed", "none"


# Re-test
test_cases = [
    ("Avenida de la Paloma, 36, 29003, Málaga, Spain", "house_number"),
    ("Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain", "house_number"),
    ("Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain", "house_number"),
    ("Andarax and Camino de San Rafael, Málaga, Spain", "intersection"),
    ("Calle Agata and Calle Colmenarejo Nuevo, Málaga, Spain", "intersection"),
    ("Calle Almogía, Málaga, Spain", "street_only"),
]

print("Re-testing with intersection fallback:\n")
for addr, addr_type in test_cases:
    print(f"[{addr_type}]  {addr}")
    loc, quality, method = geocode_with_fallbacks(addr, addr_type)
    if loc:
        print(f"  ✓ ({loc.latitude:.5f}, {loc.longitude:.5f})  quality={quality}, method={method}")
        print(f"    {loc.address[:120]}")
    else:
        print(f"  ✗ FAILED")
    print()

Re-testing with intersection fallback:

[house_number]  Avenida de la Paloma, 36, 29003, Málaga, Spain
  ✓ (36.70166, -4.44376)  quality=high, method=simplified
    36, Avenida Paloma, San Carlos Condote, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, Espa

[house_number]  Avenida de la Paloma, 36-38, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70066, -4.44184)  quality=medium, method=simplified
    Avenida Paloma, Girón, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29003, España

[house_number]  Calle Goya, 7, Carretera de Cádiz, 29002, Málaga, Spain
  ✓ (36.70553, -4.43741)  quality=high, method=original
    7, Calle Goya, Huelin, Carretera de Cádiz, Málaga, Málaga-Costa del Sol, Málaga, Andalucía, 29002, España

[house_number]  Calle Federico García Lorca, 13-11, Carretera de Cádiz, 29003, Málaga, Spain
  ✓ (36.70023, -4.43988)  quality=medium, method=original
    Calle Federico García Lorca, 25 Años de Paz, Carretera de Cádi

In [16]:
import json
from datetime import datetime

# Path for incremental results (so we can resume if interrupted)
CACHE_PATH = DATA_INTERIM / "geocoding_cache.json"

# Load existing cache if present
if CACHE_PATH.exists():
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        cache = json.load(f)
    print(f"Loaded {len(cache)} cached results from previous run")
else:
    cache = {}
    print("No cache found — starting fresh")


def save_cache():
    """Persist the cache to disk."""
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# Process all addresses
total = len(df)
already_done = sum(1 for addr in df["address_clean"] if addr in cache)
print(f"\nTotal addresses: {total}")
print(f"Already in cache: {already_done}")
print(f"To process now: {total - already_done}")
print(f"\nEstimated time: ~{(total - already_done) * 2 / 60:.1f} minutes\n")

start_time = datetime.now()

for i, row in df.iterrows():
    addr = row["address_clean"]
    addr_type = row["address_type"]
    
    # Skip if cached
    if addr in cache:
        continue
    
    # Geocode with fallbacks
    loc, quality, method = geocode_with_fallbacks(addr, addr_type)
    
    if loc:
        cache[addr] = {
            "lat": loc.latitude,
            "lon": loc.longitude,
            "quality": quality,
            "method": method,
            "matched_address": loc.address,
            "match_class": loc.raw.get("class"),
            "match_type": loc.raw.get("type"),
        }
    else:
        cache[addr] = {
            "lat": None,
            "lon": None,
            "quality": "failed",
            "method": "none",
            "matched_address": None,
            "match_class": None,
            "match_type": None,
        }
    
    # Progress: print every 25 addresses + save cache
    if (i + 1) % 25 == 0:
        elapsed = (datetime.now() - start_time).total_seconds()
        per_addr = elapsed / max(1, (i + 1 - already_done))
        eta = per_addr * (total - i - 1) / 60
        print(f"  [{i+1:3d}/{total}] cached, elapsed {elapsed:.0f}s, ETA {eta:.1f} min")
        save_cache()

# Final save
save_cache()

elapsed = (datetime.now() - start_time).total_seconds()
print(f"\n✓ Done! Processed {total} addresses in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"  Cache file: {CACHE_PATH}")

Loaded 418 cached results from previous run

Total addresses: 478
Already in cache: 418
To process now: 60

Estimated time: ~2.0 minutes

  [275/478] cached, elapsed 52s, ETA 176.1 min

✓ Done! Processed 478 addresses in 105s (1.7 min)
  Cache file: c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\geocoding_cache.json


In [17]:
# Reload the freshly cleaned addresses from notebook 01
df = pd.read_csv(INPUT_PATH)
current_addresses = set(df["address_clean"])

# Reload existing cache
with open(CACHE_PATH, "r", encoding="utf-8") as f:
    cache = json.load(f)

print(f"Cache before cleanup: {len(cache)} addresses")
print(f"Current cleaned addresses: {len(current_addresses)}")

# Find cache entries that no longer match any current address (these are stale, e.g. 'Avenidaenida...')
stale_keys = [k for k in cache.keys() if k not in current_addresses]
print(f"\nStale cache entries to remove: {len(stale_keys)}")

# Show a few examples to confirm we're removing what we think
print("\nSample of stale entries (first 5):")
for k in stale_keys[:5]:
    print(f"  → {k}")

# Remove them
for k in stale_keys:
    del cache[k]

# Save cleaned cache
save_cache()

# Report what's left to do
in_cache = sum(1 for addr in df["address_clean"] if addr in cache)
to_process = len(df) - in_cache
print(f"\nAfter cleanup: {len(cache)} cached, {to_process} still to geocode")

Cache before cleanup: 478 addresses
Current cleaned addresses: 478

Stale cache entries to remove: 0

Sample of stale entries (first 5):

After cleanup: 478 cached, 0 still to geocode


In [25]:
# Convert cache to a dataframe and merge with original
results_records = []
for addr in df["address_clean"]:
    if addr in cache:
        results_records.append(cache[addr])
    else:
        results_records.append({"lat": None, "lon": None, "quality": "missing", "method": "none"})

df_geocoded = df.copy()
df_geocoded["lat"] = [r.get("lat") for r in results_records]
df_geocoded["lon"] = [r.get("lon") for r in results_records]
df_geocoded["quality"] = [r.get("quality") for r in results_records]
df_geocoded["method"] = [r.get("method") for r in results_records]
df_geocoded["matched_address"] = [r.get("matched_address") for r in results_records]

# Overall quality distribution
print("=== Geocoding quality distribution ===")
print(df_geocoded["quality"].value_counts())
print(f"\nSuccess rate: {(df_geocoded['quality'] != 'failed').sum() / len(df_geocoded) * 100:.1f}%")

# Quality by address type
print("\n=== Quality by original address type ===")
print(pd.crosstab(df_geocoded["address_type"], df_geocoded["quality"], margins=True))

# Method distribution (for successful ones)
print("\n=== Methods used (for successful geocoding) ===")
print(df_geocoded.loc[df_geocoded["quality"] != "failed", "method"].value_counts())

# Show 5 failed addresses
print("\n=== Sample of FAILED addresses ===")
failed = df_geocoded[df_geocoded["quality"] == "failed"]
print(f"Total failed: {len(failed)}")
for _, row in failed.head(10).iterrows():
    print(f"  [{row['address_type']}]  {row['address_clean']}")

=== Geocoding quality distribution ===
quality
high      196
medium    132
low       105
failed     45
Name: count, dtype: int64

Success rate: 90.6%

=== Quality by original address type ===
quality       failed  high  low  medium  All
address_type                                
house_number      36   170   43     132  381
intersection       3     0   61       0   64
street_only        6    26    1       0   33
All               45   196  105     132  478

=== Methods used (for successful geocoding) ===
method
original                         298
first_street_of_intersection      56
generic_and_fallback              36
simplified                        23
street_only_fallback               8
second_street_of_intersection      4
first_street_with_number           2
no_district                        1
extra_abbrev_original              1
expanded_names                     1
typo_fix                           1
extra_abbrev_simplified            1
first_street_simplified            1
N

In [21]:
def expand_name_abbreviations(s: str) -> str:
    """Expand abbreviations inside street names, not just street types."""
    rules = [
        (r"\bSgto\.?", "Sargento"),
        (r"\bGral\.?", "General"),
        (r"\bDr\.?",   "Doctor"),
        (r"\bDra\.?",  "Doctora"),
        (r"\bNtra\.?\s*Sra\.?", "Nuestra Señora"),
        (r"\bSta\.?",  "Santa"),
        # "S." followed by space and a capital letter = San
        (r"\bS\.\s+(?=[A-ZÁÉÍÓÚÑ])", "San "),
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    return s


def simplify_address(address: str) -> str:
    """Remove 'de la', 'del', 'de' between street type and street name.
    Supports a wide set of street types (including Carril, Alameda, etc)."""
    return re.sub(
        r"\b(Avenida|Calle|Plaza|Camino|Carretera|Pasaje|Paseo|Glorieta|Carril|Alameda|Cuesta|Travesía|Pje)\s+(de\s+la|de\s+las|del|de\s+los|de)\s+",
        r"\1 ",
        address,
        flags=re.IGNORECASE,
    )


def strip_district(address: str) -> str:
    """Remove a 'district' chunk between house number and postal code.
    Example: 'Calle X, 26, Distrito Centro, 29009, Málaga, Spain'
          → 'Calle X, 26, 29009, Málaga, Spain'
    """
    return re.sub(
        r"(,\s*\d+[\d\-A-Za-z]*),\s*[^,\d]+(?=,\s*\d{5}\b)",
        r"\1",
        address,
    )


def strip_house_number(address: str) -> str:
    """Remove the first numeric block (house number) from an address."""
    return re.sub(r",?\s*\d+[\d\-A-Za-z]*\s*,", ",", address, count=1).strip(", ")


def reduce_house_range(address: str) -> str:
    """Convert house number ranges like '13-11' or '6-8' to just '13' or '6'."""
    return re.sub(r"(,\s*)(\d+)-\d+", r"\g<1>\g<2>", address)


def split_intersection(address: str):
    """If the address contains ' and ', return the first and second street names."""
    parts = re.split(r"\s+and\s+", address, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) == 2:
        street_a = parts[0].strip()
        street_b_raw = parts[1].strip()
        street_b_name = street_b_raw.split(",")[0].strip()
        return street_a + ", Málaga, Spain", street_b_name + ", Málaga, Spain"
    return None, None


def split_intersection_with_number(address: str):
    """For mixed addresses like 'Street A and Street B, 3, Málaga, Spain'."""
    m = re.search(r"^(.+?\s+and\s+.+?),\s*(\d+),\s*(.+)$", address, flags=re.IGNORECASE)
    if m:
        full_intersection = f"{m.group(1)}, {m.group(3)}"
        first_with_number = f"{m.group(1).split(' and ')[0]}, {m.group(2)}, {m.group(3)}"
        return full_intersection, first_with_number
    return None, None


# Manual replacements for known typos
MANUAL_TYPO_FIXES = {
    "Petersen": "Peterson",
    "Herrea":   "Herrera",
}

def apply_typo_fixes(address: str) -> str:
    """Apply known manual typo corrections."""
    for wrong, right in MANUAL_TYPO_FIXES.items():
        address = re.sub(rf"\b{wrong}\b", right, address, flags=re.IGNORECASE)
    return address


# Updated fallback chain with all v3 improvements
def geocode_with_fallbacks_v3(address_clean: str, address_type: str):
    """Extended fallback strategy v3."""
    
    # Attempt 1: original
    loc = geocode_in_malaga(address_clean)
    if loc:
        if loc.raw.get("class") == "place" and loc.raw.get("type") == "house":
            return loc, "high", "original"
        elif address_type == "street_only":
            return loc, "high", "original"
        else:
            return loc, "medium", "original"
    
    # Attempt 2: typo fixes
    fixed = apply_typo_fixes(address_clean)
    if fixed != address_clean:
        loc = geocode_in_malaga(fixed)
        if loc:
            if loc.raw.get("class") == "place" and loc.raw.get("type") == "house":
                return loc, "high", "typo_fix"
            elif address_type == "street_only":
                return loc, "high", "typo_fix"
            else:
                return loc, "medium", "typo_fix"
    
    # Attempt 3: expand name abbreviations
    expanded = expand_name_abbreviations(fixed)
    if expanded != fixed:
        loc = geocode_in_malaga(expanded)
        if loc:
            if loc.raw.get("class") == "place" and loc.raw.get("type") == "house":
                return loc, "high", "expanded_names"
            else:
                return loc, "medium", "expanded_names"
    
    # Attempt 4: strip district between number and postcode
    no_district = strip_district(expanded if expanded != fixed else fixed)
    if no_district != (expanded if expanded != fixed else fixed):
        loc = geocode_in_malaga(no_district)
        if loc:
            if loc.raw.get("class") == "place" and loc.raw.get("type") == "house":
                return loc, "high", "no_district"
            else:
                return loc, "medium", "no_district"
    
    # Attempt 5: simplified
    base_for_simplify = no_district if no_district != (expanded if expanded != fixed else fixed) else (expanded if expanded != fixed else fixed)
    simplified = simplify_address(base_for_simplify)
    if simplified != address_clean:
        loc = geocode_in_malaga(simplified)
        if loc:
            if loc.raw.get("class") == "place" and loc.raw.get("type") == "house":
                return loc, "high", "simplified"
            elif address_type == "street_only":
                return loc, "high", "simplified"
            else:
                return loc, "medium", "simplified"
    
    # Attempt 6: reduce house number range
    reduced = reduce_house_range(address_clean)
    if reduced != address_clean:
        loc = geocode_in_malaga(reduced)
        if loc:
            if loc.raw.get("class") == "place" and loc.raw.get("type") == "house":
                return loc, "high", "reduced_range"
            else:
                return loc, "medium", "reduced_range"
    
    # Attempt 7: mixed intersection with number
    full_intersection, first_with_number = split_intersection_with_number(address_clean)
    if full_intersection:
        loc = geocode_in_malaga(full_intersection)
        if loc:
            return loc, "low", "intersection_no_number"
        loc = geocode_in_malaga(first_with_number)
        if loc:
            return loc, "medium", "first_street_with_number"
    
    # Attempt 8: pure intersection
    if address_type == "intersection":
        street_a, street_b = split_intersection(address_clean)
        if street_a:
            for variant in [street_a, simplify_address(street_a), street_b, simplify_address(street_b) if street_b else None]:
                if variant:
                    loc = geocode_in_malaga(variant)
                    if loc:
                        return loc, "low", "intersection_fallback"
    
    # Attempt 9: for house_number, strip number entirely
    if address_type == "house_number":
        no_number = strip_house_number(address_clean)
        if no_number != address_clean:
            for variant in [no_number, simplify_address(no_number), expand_name_abbreviations(no_number), apply_typo_fixes(no_number)]:
                if variant and variant != address_clean:
                    loc = geocode_in_malaga(variant)
                    if loc:
                        return loc, "low", "street_only_fallback"
    
    return None, "failed", "none"


# Re-attempt currently failed addresses
failed_addresses = [
    addr for addr in df["address_clean"]
    if cache.get(addr, {}).get("quality") == "failed"
]
print(f"Re-attempting {len(failed_addresses)} failed addresses with v3 logic\n")

start_time = datetime.now()
new_successes = 0

for i, addr in enumerate(failed_addresses):
    addr_type = df.loc[df["address_clean"] == addr, "address_type"].iloc[0]
    loc, quality, method = geocode_with_fallbacks_v3(addr, addr_type)
    
    if loc:
        cache[addr] = {
            "lat": loc.latitude,
            "lon": loc.longitude,
            "quality": quality,
            "method": method,
            "matched_address": loc.address,
            "match_class": loc.raw.get("class"),
            "match_type": loc.raw.get("type"),
        }
        new_successes += 1
    
    if (i + 1) % 10 == 0:
        elapsed = (datetime.now() - start_time).total_seconds()
        print(f"  [{i+1:3d}/{len(failed_addresses)}] retried, +{new_successes} new successes, elapsed {elapsed:.0f}s")
        save_cache()

save_cache()
elapsed = (datetime.now() - start_time).total_seconds()
print(f"\n✓ Done! +{new_successes} new successes, still failed: {len(failed_addresses) - new_successes}")

Re-attempting 90 failed addresses with v3 logic

  [ 10/90] retried, +6 new successes, elapsed 36s
  [ 20/90] retried, +6 new successes, elapsed 63s
  [ 30/90] retried, +6 new successes, elapsed 85s
  [ 40/90] retried, +6 new successes, elapsed 104s
  [ 50/90] retried, +7 new successes, elapsed 144s
  [ 60/90] retried, +7 new successes, elapsed 175s
  [ 70/90] retried, +7 new successes, elapsed 210s
  [ 80/90] retried, +7 new successes, elapsed 266s


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Calle Salistre, Málaga, Spain',), **{'country_codes': 'es', 'viewbox': [(36.65, -4.55), (36.78, -4.3)], 'bounded': True}).
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 1319, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              encode_chunked=req.has_header('Transfer-encoding'))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 1358, in request
    self._send_request(method, url, body, headers, encode_chunked)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 1404, in _send_request
    self.endheaders(body, encode_chunked=encode_chunked

  [ 90/90] retried, +7 new successes, elapsed 331s

✓ Done! +7 new successes, still failed: 83


In [23]:
# Look at how the problematic addresses are stored after cleaning
samples = [
    "Cmo de S. Rafael",
    "Lebrija",
    "Antonio Machado",
    "Plaza Babel",
    "Plaza Jose Begamin",
]

for keyword in samples:
    matches = df[df["address_clean"].str.contains(keyword, case=False, na=False)]
    print(f"=== Containing '{keyword}' ===")
    for _, row in matches.iterrows():
        print(f"  raw:   {row['address_raw']}")
        print(f"  clean: {row['address_clean']}")
        print(f"  type:  {row['address_type']}")
        print()

=== Containing 'Cmo de S. Rafael' ===
  raw:   Cmo de S. Rafael 106
  clean: Cmo de S. Rafael 106, Málaga, Spain
  type:  house_number

=== Containing 'Lebrija' ===
  raw:   C. Lebrija, 6-8 malaga
  clean: Calle Lebrija, 6-8, Málaga, Spain
  type:  house_number

=== Containing 'Antonio Machado' ===
  raw:   P.º de Antonio Machado, 64
  clean: P.º de Antonio Machado, 64, Málaga, Spain
  type:  house_number

=== Containing 'Plaza Babel' ===
  raw:   plaza babel
  clean: Plaza Babel, Málaga, Spain
  type:  street_only

=== Containing 'Plaza Jose Begamin' ===
  raw:   plaza Jose Begamin, 12 - malaga
  clean: Plaza Jose Begamin, 12, Málaga, Spain
  type:  house_number



In [24]:
def apply_extra_abbreviations(s: str) -> str:
    """Fix abbreviations that slipped through the main cleaning step in notebook 01."""
    rules = [
        (r"\bCmo\.?\s",     "Camino "),   # 'Cmo de ...' → 'Camino de ...'
        (r"\bP\.º\s",       "Paseo "),    # 'P.º de ...' → 'Paseo de ...'
        (r"\bP\.\º\s",      "Paseo "),    # alternate spelling
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s)
    return s


def try_intersection_fallback_generic(address: str):
    """Generic intersection splitter — works for ANY address containing ' and '.
    Returns first found location, or None.
    """
    if " and " not in address.lower():
        return None
    
    # Split on first ' and '
    parts = re.split(r"\s+and\s+", address, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) != 2:
        return None
    
    street_a_raw = parts[0].strip()
    street_b_raw = parts[1].strip()
    
    # Try to extract clean street A (with everything before ' and ', append Málaga)
    # Remove any leading number from street_b (e.g. "11 and Calle X" → just "Calle X")
    street_b_name = re.sub(r"^\d+[\d\-A-Za-z]*\s*", "", street_b_raw)
    street_b_name = street_b_name.split(",")[0].strip()
    
    candidates = [
        f"{street_a_raw}, Málaga, Spain",
        f"{simplify_address(street_a_raw)}, Málaga, Spain",
        f"{apply_extra_abbreviations(street_a_raw)}, Málaga, Spain",
        f"{street_b_name}, Málaga, Spain",
        f"{simplify_address(street_b_name)}, Málaga, Spain",
    ]
    # Deduplicate while preserving order
    seen = set()
    candidates = [c for c in candidates if not (c in seen or seen.add(c))]
    
    for candidate in candidates:
        loc = geocode_in_malaga(candidate)
        if loc:
            return loc
    return None


def geocode_with_fallbacks_v4(address_clean: str, address_type: str):
    """Final fallback strategy: v3 + extra abbreviations + generic and-fallback."""
    
    # Pre-process with extra abbreviations
    address_pre = apply_extra_abbreviations(address_clean)
    
    # Try the full v3 chain on the pre-processed address
    loc, quality, method = geocode_with_fallbacks_v3(address_pre, address_type)
    if loc:
        # If preprocessing changed the address, mark the method
        if address_pre != address_clean:
            method = f"extra_abbrev_{method}"
        return loc, quality, method
    
    # Generic and-fallback for any address still containing 'and'
    loc = try_intersection_fallback_generic(address_clean)
    if loc:
        return loc, "low", "generic_and_fallback"
    
    # Also try after applying extra abbreviations
    if address_pre != address_clean:
        loc = try_intersection_fallback_generic(address_pre)
        if loc:
            return loc, "low", "extra_abbrev_generic_and_fallback"
    
    return None, "failed", "none"


# Re-attempt currently failed addresses
failed_addresses = [
    addr for addr in df["address_clean"]
    if cache.get(addr, {}).get("quality") == "failed"
]
print(f"Re-attempting {len(failed_addresses)} failed addresses with v4 logic\n")

start_time = datetime.now()
new_successes = 0

for i, addr in enumerate(failed_addresses):
    addr_type = df.loc[df["address_clean"] == addr, "address_type"].iloc[0]
    loc, quality, method = geocode_with_fallbacks_v4(addr, addr_type)
    
    if loc:
        cache[addr] = {
            "lat": loc.latitude,
            "lon": loc.longitude,
            "quality": quality,
            "method": method,
            "matched_address": loc.address,
            "match_class": loc.raw.get("class"),
            "match_type": loc.raw.get("type"),
        }
        new_successes += 1
    
    if (i + 1) % 10 == 0:
        elapsed = (datetime.now() - start_time).total_seconds()
        print(f"  [{i+1:3d}/{len(failed_addresses)}] retried, +{new_successes} new successes, elapsed {elapsed:.0f}s")
        save_cache()

save_cache()
elapsed = (datetime.now() - start_time).total_seconds()
print(f"\n✓ Done! +{new_successes} new successes, still failed: {len(failed_addresses) - new_successes}")

Re-attempting 83 failed addresses with v4 logic

  [ 10/83] retried, +7 new successes, elapsed 32s
  [ 20/83] retried, +13 new successes, elapsed 62s
  [ 30/83] retried, +21 new successes, elapsed 99s
  [ 40/83] retried, +29 new successes, elapsed 141s
  [ 50/83] retried, +35 new successes, elapsed 178s


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Casabermeja, 56, 15, Málaga, Spain',), **{'country_codes': 'es', 'viewbox': [(36.65, -4.55), (36.78, -4.3)], 'bounded': True}).
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 1319, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
              encode_chunked=req.has_header('Transfer-encoding'))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 1358, in request
    self._send_request(method, url, body, headers, encode_chunked)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\http\client.py", line 1404, in _send_request
    self.endheaders(body, encode_chunked=encode_ch

GeocoderTimedOut: Service timed out

In [26]:
# Build the final geocoded dataframe from cache
results = []
for _, row in df.iterrows():
    addr = row["address_clean"]
    cache_entry = cache.get(addr, {})
    results.append({
        "address_raw": row["address_raw"],
        "address_clean": addr,
        "address_type": row["address_type"],
        "lat": cache_entry.get("lat"),
        "lon": cache_entry.get("lon"),
        "geocoding_quality": cache_entry.get("quality", "failed"),
        "geocoding_method": cache_entry.get("method", "none"),
        "matched_address": cache_entry.get("matched_address"),
        "match_class": cache_entry.get("match_class"),
        "match_type": cache_entry.get("match_type"),
    })

df_final = pd.DataFrame(results)

# Save
df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved {len(df_final)} rows to {OUTPUT_PATH}")
print(f"  Successfully geocoded: {(df_final['geocoding_quality'] != 'failed').sum()}")
print(f"  Failed: {(df_final['geocoding_quality'] == 'failed').sum()}")

df_final.head(5)

Saved 478 rows to c:\Users\user\Documents\Projects\malaga-textile-access\data\processed\containers_geocoded.csv
  Successfully geocoded: 433
  Failed: 45


,address_raw,address_clean,address_type,lat,lon,geocoding_quality,geocoding_method,matched_address,match_class,match_type
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...","Avenida de la Paloma, 36-38, Carretera de Cádi...",house_number,36.700663,-4.441837,medium,simplified,"Avenida Paloma, Girón, Carretera de Cádiz, Mál...",highway,secondary
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...","Avenida de la Paloma, 43, Carretera de Cádiz, ...",house_number,36.702144,-4.445557,high,simplified,"43, Avenida Paloma, San Carlos Condote, Carret...",place,house
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...","Calle Conde de Guadalhorce, 6-8, Cruz de Humil...",house_number,36.714611,-4.442747,medium,original,"Calle Conde de Guadalhorce, Cruz del Humillade...",highway,secondary
3,"C. Federico García Lorca, 13-11, Carretera de ...","Calle Federico García Lorca, 13-11, Carretera ...",house_number,36.700226,-4.439883,medium,original,"Calle Federico García Lorca, 25 Años de Paz, C...",highway,residential
4,"Calle Goya, 7, Carretera de Cádiz, 29002 Málaga","Calle Goya, 7, Carretera de Cádiz, 29002, Mála...",house_number,36.705526,-4.437411,high,original,"7, Calle Goya, Huelin, Carretera de Cádiz, Mál...",place,house


In [27]:
# Sanity check: are all points actually in Málaga?

# Filter to successfully geocoded
df_geo = df_final[df_final["geocoding_quality"] != "failed"].copy()
print(f"Total geocoded points: {len(df_geo)}")

# Basic stats
print(f"\nLatitude:  min={df_geo['lat'].min():.5f}, max={df_geo['lat'].max():.5f}")
print(f"Longitude: min={df_geo['lon'].min():.5f}, max={df_geo['lon'].max():.5f}")

# Málaga is roughly centered at (36.72, -4.42)
# A reasonable bounding box for the city: lat 36.65-36.78, lon -4.55 to -4.30
MALAGA_LAT_MIN, MALAGA_LAT_MAX = 36.65, 36.78
MALAGA_LON_MIN, MALAGA_LON_MAX = -4.55, -4.30

# Flag outliers
outliers = df_geo[
    (df_geo["lat"] < MALAGA_LAT_MIN) | (df_geo["lat"] > MALAGA_LAT_MAX) |
    (df_geo["lon"] < MALAGA_LON_MIN) | (df_geo["lon"] > MALAGA_LON_MAX)
]
print(f"\nPoints outside Málaga bbox: {len(outliers)}")

if len(outliers) > 0:
    print("\nOutlier details:")
    for _, row in outliers.iterrows():
        print(f"  ({row['lat']:.5f}, {row['lon']:.5f})  quality={row['geocoding_quality']}")
        print(f"    raw:     {row['address_raw']}")
        print(f"    clean:   {row['address_clean']}")
        print(f"    matched: {row['matched_address']}")
        print()

Total geocoded points: 433

Latitude:  min=36.65329, max=36.75546
Longitude: min=-4.54948, max=-4.34496

Points outside Málaga bbox: 0
